# Visualize Data Egress from S3 Buckets

This notebook queries CloudTrail logs to retrieve S3 `GetObject` requests and visualize the results as a graph. 

Source nodes are S3 buckets, and destination nodes are clients that have downloaded data.

The larger the node, the more data has been downloaded to/from the node.

First, we install dependencies.

In [ ]:
%pip install https://scanner-dev-public.s3.us-west-2.amazonaws.com/sdks/python/scanner_client-0.0.1-py3-none-any.whl
%pip install yfiles_jupyter_graphs pandas

In [ ]:
import os
import pandas as pd
from scanner_client import Scanner
from yfiles_jupyter_graphs import GraphWidget
from datetime import datetime, timezone, timedelta

Add utility function to convert Scanner search results to a `pandas` data frame.

In [ ]:
def convert_results_to_data_frame(results):
    rows = [row.columns.to_dict() for row in results.rows]
    column_tags = results.column_tags.to_dict()
    if len(column_tags) > 0:
        # If this is a table, use the column ordering in the data frame
        return pd.DataFrame(data=rows, columns=results.column_ordering)
    else:
        # Otherwise, this is a list of log events, so use pandas JSON
        # normalization to set the table columns to the union of all keys.
        return pd.json_normalize(rows)


Initialize Scanner API client:

In [ ]:
scanner = Scanner(
    api_url=os.environ["SCANNER_API_URL"],
    api_key=os.environ["SCANNER_API_KEY"],
)

Set analyzed time range.

In [ ]:
end_time = datetime.now(tz=timezone.utc)
start_time = end_time - timedelta(days=7)

Query for `GetObject` requests, aggregating statistics by source S3 bucket and destination client IP address.

In [ ]:
response = scanner.query.blocking_query(
    start_time=start_time.isoformat(),
    end_time=end_time.isoformat(),
    query_text="""
        %ingest.source_type: "aws:cloudtrail"
        eventName: "GetObject"
        eventSource: "s3.amazonaws.com"
        | rename 
          requestParameters.bucketName as srcS3Bucket, 
          sourceIPAddress as dstIP,
          userIdentity.type as dstUserType,
          additionalEventData.bytesTransferredOut as bytesTransferredOut
        | stats 
          sum(bytesTransferredOut) as totalBytesTransferredOut
          by 
          srcS3Bucket,
          dstUserType,
          dstIP 
    """
)
len(response.results.rows)

In [ ]:
df = convert_results_to_data_frame(response.results)

In [ ]:
df.head()

Compute nodes and edges of S3 data egress operations.

Source nodes are S3 buckets.
Destination nodes are clients.

In [ ]:
from collections import defaultdict

MB = (1 << 20)

node_total_mb_transferred = defaultdict(int)
edges = []
node_ids = set()
for i, row in enumerate(df.to_dict(orient="records")):
    if not row['srcS3Bucket'] or not row['dstIP']:
        next
    src_key = row['srcS3Bucket']
    dst_key = f"{row['dstIP']}/{row['dstUserType']}"
    node_ids.add(src_key)
    node_ids.add(dst_key)
    transferred_mb = row.get('totalBytesTransferredOut', 0) / MB
    node_total_mb_transferred[row['srcS3Bucket']] += transferred_mb
    node_total_mb_transferred[row['dstIP']] += transferred_mb
    edges.append({
        'id': i,
        'start': src_key,
        'end': dst_key,
        'properties': {
            'source_s3_bucket': row['srcS3Bucket'],
            'destination_IP': row['dstIP'],
            'destination_user_type': row['dstUserType'],
            'label': str(transferred_mb),
            'transferred_mb': transferred_mb,
        }
    })

nodes = []
for node_id in node_ids:
    if not node_id:
        continue
    transferred_mb = node_total_mb_transferred[node_id]
    parts = node_id.split('/')
    properties = {
        'label': node_id,
        'transferred_mb': transferred_mb,
    }
    if len(parts) == 1:
        # s3 bucket
        properties['s3_bucket'] = node_id
    elif len(parts) == 2:
        properties['ip_address'] = parts[0]
        properties['user_type'] = parts[1]
        
    nodes.append({
        'id': node_id,
        'properties': properties,
    })

# Set node size based on the number of times it is involved in an AssumeRole operation.
min_transferred_mb = min(node_total_mb_transferred.values())
max_transferred_mb = max(node_total_mb_transferred.values())

min_scale_factor = 1.0
max_scale_factor = 10.0

def scale_factor_mapping(node):
    transferred_mb = node['properties']['transferred_mb']
    numer = transferred_mb - min_transferred_mb
    denom = max_transferred_mb - min_transferred_mb
    if denom == 0:
        return 1.0
    frac = numer / denom
    delta = (max_scale_factor - min_scale_factor) * frac
    return min_scale_factor + delta

Generate interactive graph visualization of S3 data egress.
- Click and drag to navigate. Use mouse wheel to zoom in/out.
- Click on a node or an edge to select it and see what it is connected to.
- When a node or edge is selected, inspect its properties in the `Data` tab in the side bar.
- Search for a node or edge via the `Search` tab in the side bar.
- Change the layout of the graph to examine relationships in different ways.

In [ ]:
w = GraphWidget()
w.nodes = nodes
w.edges = edges
w.directed = True
w.set_node_scale_factor_mapping(scale_factor_mapping)
w